# Boston 311 Intake Assistant (RAG + Structured Output)

**Objective:** Take a resident's plain-language message and return the correct Boston 311 request type, department, and queue, as JSON. If the input message is unclear, the assistant asks a follow-up question.

**Pipeline:**
1. Load the data and keep the four in-scope departments.
2. Build a de-duplicated reference list of real request types with their department and queue.
3. Embed the reference list.
4. For each message, retrieve the top K closest types, then ask the LLM to pick one and return JSON.
5. Test on a set of messages and report accuracy.

**AI use disclosure:** The test messages in step 5 were drafted with the help of Claude and then checked against the real reference list.

## 1. Setup

In [5]:
import os, re, html, json, getpass
from pathlib import Path
from typing import Optional, Literal

import numpy as np
import pandas as pd
from pydantic import BaseModel
from openai import OpenAI

REPO_DIR = Path.cwd()
if REPO_DIR.name == "notebooks" and REPO_DIR.parent.name == "genai_assistant":
    REPO_DIR = REPO_DIR.parent.parent
elif REPO_DIR.name == "genai_assistant":
    REPO_DIR = REPO_DIR.parent

RAW_DATA_DIR = REPO_DIR / "data" / "raw"
RESULTS_DIR = REPO_DIR / "genai_assistant" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

files = ["lagan_311_open_data_2024_rvsd3.csv", "lagan_311_open_data_2023_rvsd3.csv"]
folders = [RAW_DATA_DIR, Path("/content"), Path(".")]
paths = []

for filename in files:
    for folder in folders:
        p = folder / filename
        if p.exists() and p.resolve() not in [Path(x).resolve() for x in paths]:
            paths.append(p)

DEPT_CODES = ["PWDx", "BTDT", "PARK", "ISD"]
CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"
TOP_K = 8          # number of candidate types retriever provides to LLM
MIN_CASES = 20     # drop rare types from reference list

if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

client = OpenAI()

## 2. Load the Boston 311 data
Read just the columns we need since the full file is too large and keep the four departments defined in the cell above. Drop the rows without a type or queue as they can't be routed.

In [11]:
USECOLS = ["case_title", "type", "reason", "subject", "department", "queue", "source"]

if not paths:
    raise FileNotFoundError(f"Could not find {files}. Place both files in {RAW_DATA_DIR}.")

rows = pd.concat(
    [pd.read_csv(p, usecols=USECOLS, dtype=str, low_memory=False) for p in paths],
    ignore_index=True,
)

print("Files loaded:", paths)
print("All rows:", len(rows))

rows = rows[rows["department"].isin(DEPT_CODES)].dropna(subset=["type", "queue"]).copy()
print("Rows with a type and queue:", len(rows))
print('\n')
print(rows["department"].value_counts())

Files loaded: ['/content/lagan_311_open_data_2024_rvsd3.csv', '/content/lagan_311_open_data_2023_rvsd3.csv']
All rows: 614547
Rows with a type and queue: 558663


department
PWDx    328878
BTDT    156761
ISD      36747
PARK     36277
Name: count, dtype: int64


## 3. Build the searchable reference list
case_title is messy and has many variants, HTML codes like `&amp;`, internal codes, etc, and in the raw data, it mostly just repeats the type name (for example 119,806 of 119,807 Parking Enforcement titles). The reference list is built at the type level instead, after taking care of whitespace and HTML codes. For each type we keep:
- most common department and queue
- reason and subject
- the share of requests that came from residents as compared to city employees

In [12]:
# fix any double spaces
def clean_title(t):
    t = html.unescape(str(t)).strip()
    return re.sub(r"\s+", " ", t)

rows["type"] = rows["type"].map(clean_title)
rows["is_resident"] = rows["source"].fillna("").ne("Employee Generated")
resident = rows[rows["is_resident"]]

# get the most common department & queue for each type
def dominant(df):
    c = df.groupby(["type", "department", "queue"]).size().rename("n").reset_index()
    return c.sort_values("n", ascending=False).drop_duplicates("type").set_index("type")

best_all = dominant(rows)
best_res = dominant(resident)
all_n = rows.groupby("type").size()
res_n = resident.groupby("type").size()

def mode(s):
    s = s.dropna()
    return s.mode().iloc[0] if len(s) else ""

meta = rows.groupby("type").agg(reason=("reason", mode), subject=("subject", mode))

new_rows = []

for t, total in all_n.items():
    if total < MIN_CASES:
        continue
    use_res = res_n.get(t, 0) >= 5
    top = (best_res if use_res else best_all).loc[t]
    den = res_n[t] if use_res else total

    reason = meta.loc[t, "reason"]
    subject = meta.loc[t, "subject"]
    share = float(top["n"]/den)

    new_rows.append({
        "type": t,
        "department": top["department"],
        "queue": top["queue"],
        "queue_share": round(share, 2),
        "queue_depends_on_location": share < 0.4,
        "reason": reason,
        "subject": subject,
        "total_cases": int(total),
        "resident_share": round(float(res_n.get(t, 0) / total), 2),
        "embed_text": f"{t}. Category: {reason}/{subject}.",
    })

ref = pd.DataFrame(new_rows).sort_values(["department", "type"]).reset_index(drop=True)
ref["ref_id"] = ref.index
ref.drop(columns="embed_text").to_csv(RESULTS_DIR / "reference_list.csv", index=False)

print("Reference list size:", len(ref))

print(ref["department"].value_counts().to_string())

print("Request types whose queue depends on location:", int(ref["queue_depends_on_location"].sum()))
ref[["ref_id", "type", "department", "queue", "queue_share", "total_cases", "resident_share"]].head(15)

Reference list size: 129
department
ISD     49
PWDx    49
BTDT    19
PARK    12
Request types whose queue depends on location: 16


,ref_id,type,department,queue,queue_share,total_cases,resident_share
0,0,Abandoned Bicycle,BTDT,BTDT_Abandoned Bicycle,0.90,2210,0.61
1,1,Abandoned Vehicles,BTDT,BTDT_AVRS Interface Queue,0.98,9339,0.44
2,2,Bicycle Issues,BTDT,BTDT_BostonBikes,0.98,100,0.58
3,3,General Comments For a Program or Policy,BTDT,BTDT_General Comments,0.34,1687,0.79
4,4,General Traffic Engineering Request,BTDT,BTDT_Engineering_New Sign and Pavement Marking...,0.64,293,0.66
5,5,Missing Sign,BTDT,BTDT_Sign Shop_Sign Installation,0.80,815,0.65
6,6,Municipal Parking Lot Complaints,BTDT,BTDT_Parking Lot_Municipal Parking Lot Complaints,0.70,25,0.40
7,7,New Sign Crosswalk or Pavement Marking,BTDT,BTDT_Engineering_New Sign and Pavement Marking...,0.74,1768,0.77
8,8,No-Tow Complaint Confirmation,BTDT,BTDT_Sign Shop_Sign Installation,0.15,134,1.00
9,9,Parking Enforcement,BTDT,BTDT_Parking Enforcement,0.99,119807,0.36


**Notes:**
- This reference list includes all types that have at least 20 requests across the four departments. Some types are mostly requests by city employees (resident_share shows what fraction of the requests came from residents versus city employees). These types are still included in the list so that the true types are reflected, but we are able to filter on resident_share to view only requests by residents.
- A type's department is usually clear, but types can appear under multiple departments when employees re-route a case. Thus, each type is assigned its most common department and queue, using requests submitted by residents as this reflects which department cases are first routed to.
- Several Public Works types are spread across many district queues such as `PWDx_District 1C: Downtown`. For such types, no particular queue is correct for every request because the queue depends on where the problem is. queue_share shows how dominant the chosen queue is, and queue_depends_on_location flags the ones that are below 40%.

## 4. Embed the reference list (RAG index)
Each type's text is turned into an embedding vector. The resident's message is embedded as well and the closest types are retrieved.

In [13]:
def embed(texts, batch_size=100):
    vectors = []
    for i in range(0, len(texts), batch_size):
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch_size])
        vectors.extend(d.embedding for d in resp.data)
    arr = np.array(vectors, dtype=np.float32)
    return arr/np.linalg.norm(arr, axis=1, keepdims=True)

ref_vectors = embed(ref["embed_text"].tolist())
print("Embedding matrix:", ref_vectors.shape)

def retrieve(message, k=TOP_K):
    q = embed([message])[0]
    sims = ref_vectors @ q
    idx = np.argsort(-sims)[:k]
    return ref.iloc[idx].assign(similarity=sims[idx])

retrieve("there's a big hole on Centre St near the T stop")[["type", "department", "queue", "similarity"]]

Embedding matrix: (129, 1536)


,type,department,queue,similarity
115,Sidewalk Cover / Manhole,PWDx,PWDx_Street Light_General Lighting Request,0.388895
81,Catchbasin,PWDx,PWDx_Roadway Repair_ARP_Resurfacing,0.318002
2,Bicycle Issues,BTDT,BTDT_BostonBikes,0.297070
125,Unshoveled Sidewalk,PWDx,PWDx_Code Enforcement,0.289097
6,Municipal Parking Lot Complaints,BTDT,BTDT_Parking Lot_Municipal Parking Lot Complaints,0.282763
117,Sidewalk Repair (Make Safe),PWDx,PWDx_Sidewalk Repair,0.268413
17,Traffic Signal Repair,BTDT,BTDT_Traffic Signal_Graffiti,0.257038
13,Phone Bank Service Inquiry,BTDT,BTDT_OPC_Phone Bank Service,0.256382


## 5. Classify with structured output
The LLM takes the retrieved candidates and returns a candidate_id, so department, type, and queue is retrieved from the reference list. The schema below is enforced by the API, and it also carries the follow-up questions in case of unclear input messages.

In [14]:
class Classification(BaseModel):
    candidate_id: Optional[int]
    confidence: Literal["high", "medium", "low"]
    needs_clarification: bool            # true if the message is vague or unclear
    follow_up_question: Optional[str]    # if needs_clarification is true, short follow up question for the resident

SYSTEM_PROMPT = """You are the intake assistant for Boston 311, the City of Boston's service request system.
A resident describes a problem in plain language. You are given a numbered list of candidate request types
retrieved from the city's list of types. Pick the single best candidate by its ID.

Rules:
- Choose only from the candidate IDs provided.
- Match on what the resident wants fixed, not on incidental words (for example a street name or a T stop).
- If the message is too vague to choose, or two candidates are equally plausible, set needs_clarification to true
  and write one short, friendly follow-up question that would clarify the confusion. You may still give your best-guess ID.
- If the message is not a city service issue or none of the candidates fit, set candidate_id to null,
  needs_clarification to true, and ask what the problem is.
- If the message is clear, set needs_clarification to false and follow_up_question to null.
- confidence reflects how sure you are of the chosen candidate."""

def format_candidates(cand):
    lines = []
    for _, c in cand.iterrows():
        lines.append(f"[{c['ref_id']}] {c['type']} (category: {c['reason']}; dept {c['department']}, queue {c['queue']})")
    return "\n".join(lines)

DEFAULT_FOLLOW_UP = "Could you tell me a bit more about the problem and where it is located?"

def classify(message, k=TOP_K):
    cand = retrieve(message, k)
    user_msg = f"Resident message: {message}\n\nCandidates:\n{format_candidates(cand)}"
    res = client.chat.completions.parse(
        model=CHAT_MODEL,
        temperature=0,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_msg}],
        response_format=Classification,
    )
    out = res.choices[0].message.parsed

    valid_ids = set(cand["ref_id"])
    cid = out.candidate_id if out.candidate_id in valid_ids else None

    needs = out.needs_clarification or cid is None
    follow_up = out.follow_up_question if needs else None
    if needs and not follow_up:
        follow_up = DEFAULT_FOLLOW_UP

    row = ref.loc[cid] if cid is not None else None
    return {
        "department": None if row is None else row["department"],
        "type": None if row is None else row["type"],
        "queue": None if row is None else row["queue"],
        "queue_depends_on_location": None if row is None else bool(row["queue_depends_on_location"]),
        "confidence": out.confidence,
        "needs_clarification": needs,
        "follow_up_question": follow_up,
        "retrieved_types": cand["type"].tolist(),
    }

# if resident answers the follow-up question, combine initial and follow-up messages and classify again
def intake(message, follow_up_answer=None):
    text = message if not follow_up_answer else f"{message} Additional detail: {follow_up_answer}"
    result = classify(text)
    return {k: v for k, v in result.items() if k != "retrieved_types"}

print(json.dumps(intake("there's a big hole on Centre St near the T stop"), indent=2))

{
  "department": "PWDx",
  "type": "Catchbasin",
  "queue": "PWDx_Roadway Repair_ARP_Resurfacing",
  "queue_depends_on_location": true,
  "confidence": "high",
  "needs_clarification": false,
  "follow_up_question": null
}


### Demo: Testing with a clear message, a vague message, and a follow-up answer

In [15]:
clear_test = "somebody's car has been blocking my driveway all morning"
print("Clear message:")
print(json.dumps(intake(clear_test), indent=2))

vague_test = "something is wrong on my street"
first = intake(vague_test)
print("\nVague message:")
print(json.dumps(first, indent=2))

print("\nAfter the resident answers the follow-up question:")
print(json.dumps(intake(vague_test, follow_up_answer="the streetlight in front of my house has been out for a week"), indent=2))

Clear message:
{
  "department": "BTDT",
  "type": "Parking Enforcement",
  "queue": "BTDT_Parking Enforcement",
  "queue_depends_on_location": false,
  "confidence": "high",
  "needs_clarification": false,
  "follow_up_question": null
}

Vague message:
{
  "department": null,
  "type": null,
  "queue": null,
  "queue_depends_on_location": null,
  "confidence": "low",
  "needs_clarification": true,
  "follow_up_question": "Can you please specify what issue you're experiencing on your street?"
}

After the resident answers the follow-up question:
{
  "department": "PWDx",
  "type": "Street Light Outages",
  "queue": "PWDx_Street Light Outages",
  "queue_depends_on_location": false,
  "confidence": "high",
  "needs_clarification": false,
  "follow_up_question": null
}


## 6. Test set of 35 resident messages
Each row has the message, the expected type and department, and a flag for messages that are meant to be unclear, in which case, the expected behaviour is to ask the default follow-up question.

- The expected labels were checked against the real type names in the 2023 and 2024 datasets. In cases where two real types are near-duplicates, the label lists both types separated by | and either type counts as correct if returned.

In [16]:
# 35 test cases across all 4 departments plus some vague cases: (message, expected_type, expected_department, should_ask_followup)
test_cases = [
    # Public Works (PWDx)
    ("there's a big hole on Centre St near the T stop", "Request for Pothole Repair", "PWDx", False),
    ("my car hit a massive pothole on Blue Hill Ave and popped a tire", "Request for Pothole Repair", "PWDx", False),
    ("they never picked up the old couch I put out for bulk pickup", "Missed Trash/Recycling/Yard Waste/Bulk Item|Schedule a Bulk Item Pickup", "PWDx", False),
    ("I want to schedule a bulk item pickup for my broken dresser", "Schedule a Bulk Item Pickup|Schedule a Bulk Item Pickup SS", "PWDx", False),
    ("I need a new recycling cart, mine is cracked", "Request for Recycling Cart", "PWDx", False),
    ("the streetlight outside my building has been out for days", "Street Light Outages", "PWDx", False),
    ("a car crashed into a light pole and knocked it over", "Street Light Knock Downs", "PWDx", False),
    ("the sidewalk on my block is cracked and someone could trip", "Sidewalk Repair (Make Safe)|Sidewalk Repair", "PWDx", False),
    ("neighbors leave their trash barrels on the sidewalk all day", "Improper Storage of Trash (Barrels)", "PWDx", False),
    ("there's a dead raccoon in the middle of the road", "Pick up Dead Animal", "PWDx", False),
    ("someone spray painted graffiti on the side of my building", "PWD Graffiti|Graffiti Removal", "PWDx", False),
    ("somebody dumped a mattress and old tires in the alley", "Illegal Dumping", "PWDx", False),
    ("the sidewalk in front of the building next door was never shoveled", "Unshoveled Sidewalk", "PWDx", False),
    ("the vacant lot next door is overgrown with weeds and garbage", "Poor Conditions of Property|Trash on Vacant Lot", "PWDx", False),
    ("a manhole cover in the road is loose and clanks every time a car goes over it", "Utility Casting Repair", "PWDx", False),
    # Transportation (BTDT)
    ("a car is parked in front of my driveway and I can't get out", "Parking Enforcement", "BTDT", False),
    ("someone has been parked in the bike lane all day", "Parking Enforcement", "BTDT", False),
    ("the traffic light at my intersection is stuck on red", "Traffic Signal Inspection|Traffic Signal Repair", "BTDT", False),
    ("the stop sign on our corner got knocked down", "Sign Repair|Missing Sign", "BTDT", False),
    ("there's an abandoned car on my street that hasn't moved in months", "Abandoned Vehicles", "BTDT", False),
    ("the parking meter on my street is broken", "Parking Meter Repairs", "BTDT", False),
    # Parks & Recreation (PARK)
    ("a big tree branch fell and is blocking the path in the park", "Tree Emergencies", "PARK", False),
    ("a tree on my street looks dead and could fall on the house", "Tree Maintenance Requests|Tree Emergencies", "PARK", False),
    ("I'd like the city to plant a tree in front of my house", "New Tree Requests", "PARK", False),
    ("the swing set at the playground is broken", "Equipment Repair", "PARK", False),
    ("the grass in our neighborhood park hasn't been mowed all summer", "Ground Maintenance", "PARK", False),
    # Inspectional Services (ISD)
    ("we have no heat in our apartment and the landlord won't respond", "Heat - Excessive Insufficient", "ISD", False),
    ("my landlord won't fix the mold and water leaks in the bathroom", "Unsatisfactory Living Conditions|Chronic Dampness/Mold", "ISD", False),
    ("I keep seeing rats in my backyard", "Rodent Activity", "ISD", False),
    ("the stairs in my apartment building are broken and unsafe", "Unsafe Dangerous Conditions|Maintenance Complaint - Residential", "ISD", False),
    ("a contractor next door is doing construction without a permit", "Work w/out Permit", "ISD", False),
    # Messages that are too vague and require a follow-up
    ("something is wrong on my street", None, None, True),
    ("it's a mess outside", None, None, True),
    ("I need help", None, None, True),
    ("the city should do something about this", None, None, True),
]

tests = pd.DataFrame(test_cases, columns=["message", "expected_type", "expected_department", "expect_followup"])
print(len(tests), "test messages,", int(tests["expect_followup"].sum()), "are intentionally vague")
print(tests["expected_department"].value_counts().to_string())

35 test messages, 4 are intentionally vague
expected_department
PWDx    15
BTDT     6
PARK     5
ISD      5


The cell below validates the labels against reference_list.csv, so any typos or types that were filtered out are flagged.

In [18]:
import difflib

known_types = set(ref["type"])
type_to_dept = dict(zip(ref["type"], ref["department"]))

def alternates(label):
    return [] if label is None or (isinstance(label, float) and pd.isna(label)) else label.split("|")

def label_problems(row):
    alts = alternates(row["expected_type"])
    if not alts:
        return ""
    unknown = [a for a in alts if a not in known_types]
    if unknown:
        return "type not in reference list: " + ", ".join(unknown)
    if not any(type_to_dept[a] == row["expected_department"] for a in alts):
        return "department in data is " + "/".join(sorted({type_to_dept[a] for a in alts}))
    return ""

tests["label_problem"] = tests.apply(label_problems, axis=1)
tests["labels_ok"] = tests["label_problem"].eq("")

bad = tests[~tests["labels_ok"]]
if len(bad):
    print("Fix these labels using reference_list.csv:\n")
    for _, b in bad.iterrows():
        close = [c for a in alternates(b["expected_type"]) for c in difflib.get_close_matches(a, list(known_types), n=2, cutoff=0.4)]
        print(f"  {b['message'][:60]!r}\n    {b['label_problem']}   (closest real types: {close})")
else:
    print("All test labels match the reference list.")

All test labels match the reference list.


## 7. Evaluation
Rows with valid labels are scored for type and department accuracy. Metrics:
- Type accuracy and department accuracy for the clear messages
- Retrieval hit at K: was the correct type among the K retrieved candidates. This tells us whether a miss came from retrieval or from the LLM
- Follow-up accuracy: in case of the vague messages, did the assistant ask a question instead of guessing?

In [19]:
records = []
for _, t in tests.iterrows():
    res = classify(t["message"])
    alts = alternates(t["expected_type"])
    records.append({
        "message": t["message"],
        "expected_type": t["expected_type"],
        "expected_department": t["expected_department"],
        "expect_followup": t["expect_followup"],
        "labels_ok": t["labels_ok"],
        "pred_type": res["type"],
        "pred_department": res["department"],
        "pred_queue": res["queue"],
        "queue_depends_on_location": res["queue_depends_on_location"],
        "confidence": res["confidence"],
        "asked_followup": res["needs_clarification"],
        "follow_up_question": res["follow_up_question"],
        "retrieval_hit": any(a in res["retrieved_types"] for a in alts) if alts else None,
    })

results = pd.DataFrame(records)

scored = results[(~results["expect_followup"]) & results["labels_ok"]].copy()
scored["type_correct"] = scored.apply(lambda r: r["pred_type"] in alternates(r["expected_type"]), axis=1)
scored["dept_correct"] = scored["pred_department"] == scored["expected_department"]

vague = results[results["expect_followup"]].copy()
vague["followup_correct"] = vague["asked_followup"]

skipped = int((~results["expect_followup"]).sum()) - len(scored)
print(f"Scored clear messages: {len(scored)} (skipped {skipped} with invalid labels)")
print(f"Type accuracy: {scored['type_correct'].mean():.1%}")
print(f"Department accuracy: {scored['dept_correct'].mean():.1%}")
print(f"Retrieval hit at {TOP_K}: {scored['retrieval_hit'].astype(bool).mean():.1%}")
print(f"Follow-up on vague messages: {vague['followup_correct'].mean():.1%} ({int(vague['followup_correct'].sum())}/{len(vague)})")
print(f"Asked an unnecessary follow-up on clear messages: {int(scored['asked_followup'].sum())}/{len(scored)}")

Scored clear messages: 31 (skipped 0 with invalid labels)
Type accuracy: 96.8%
Department accuracy: 96.8%
Retrieval hit at 8: 96.8%
Follow-up on vague messages: 100.0% (4/4)
Asked an unnecessary follow-up on clear messages: 0/31


### Errors

In [21]:
# display any misclassified messages
errors = scored[~scored["type_correct"]][
    ["message", "expected_type", "pred_type", "expected_department", "pred_department", "retrieval_hit", "confidence"]
]
errors if len(errors) else print("No type errors on the scored messages.")

,message,expected_type,pred_type,expected_department,pred_department,retrieval_hit,confidence
0,there's a big hole on Centre St near the T stop,Request for Pothole Repair,Catchbasin,PWDx,PWDx,False,high


In [23]:
# accuracy by dept
scored.groupby("expected_department")[["type_correct", "dept_correct"]].mean().round(2)

,type_correct,dept_correct
expected_department,,
BTDT,1.00,1.00
ISD,1.00,1.00
PARK,1.00,1.00
PWDx,0.93,0.93


## 8. Save results of evaluation in csv file
intake_tests.csv contains each test message with its expected and predicted type, department, and queue.

In [25]:
results.to_csv(RESULTS_DIR / "intake_tests.csv", index=False)
print(f"Saved {RESULTS_DIR / 'intake_tests.csv'} with {len(results)} rows.")

Saved intake_tests.csv with 35 rows.


## 9. Limitations
- Each type is routed to its most common queue. Many Public Works types are split across district queues, so the queue largely depends on the location. The results flags this with queue_depends_on_location, and the assistant could be extended further to choose the district queue from the neighborhood or street in the message.
- Some reference entries are rarely reported by residents (this is noted through resident_share).
- The reference list is built from type, not case_title, because the title mostly repeats the type name.
- Accuracy is calculated from a small test set, so it is more of an estimate, not a guarantee.
- Possible extensions to further develop this project:
    - an LLM pass to write a one-line plain-language description of each type before embedding, for types like `Mattress_Pickup` or `CE Collection`, which have names that residents may not be aware of
    - a hybrid keyword and embedding retriever
    - a link to the staffing model so the assistant can report department load for the next month